# Lamio v2 - MoE Inference (Ornith-35B)

Custom llama.cpp fork with mmap zero-copy expert loading + MoE-aware RSS orchestration.

- **Repo**: https://github.com/Neguiolidas/Lamio
- **Model**: ornith-ai/Ornith-1.0-35B-GGUF (Q4_K_M, ~20GB)
- **Architecture**: qwen35moe (256 experts, 8 active, delta net + attention)
- **Key feature**: ExpertRouter with LRU eviction keeps RSS under budget

Storage: `/tmp/` (ephemeral, 73GB available). Model downloaded on-demand via mmap.


In [ ]:
#@title System Check
import os, subprocess, psutil

print(f'CPU cores: {os.cpu_count()}')
print(f'RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB')
print(f'RAM available: {psutil.virtual_memory().available / 1024**3:.1f} GB')
print(f'Disk free: {psutil.disk_usage("/").free / 1024**3:.1f} GB')
print(f'Disk /tmp free: {psutil.disk_usage("/tmp").free / 1024**3:.1f} GB')

import torch
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB)')
else:
    print('No GPU (CPU-only inference)')

# Create ephemeral working dirs
os.makedirs('/tmp/lamio', exist_ok=True)
os.makedirs('/tmp/models', exist_ok=True)
print('\nDirs ready: /tmp/lamio, /tmp/models')


In [ ]:
#@title Clone Lamio + Build ggml + Lamio v2
import subprocess, os, time

REPO = 'https://github.com/Neguiolidas/Lamio.git'
LAMIO_DIR = '/tmp/lamio'

# Clone (or pull if exists)
if not os.path.exists(f'{LAMIO_DIR}/.git'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, LAMIO_DIR], check=True)
else:
    subprocess.run(['git', '-C', LAMIO_DIR, 'pull'], capture_output=True)

print(f'Repo at {LAMIO_DIR}')
subprocess.run(['git', '-C', LAMIO_DIR, 'log', '--oneline', '-5'], check=True)

# Build ggml (CPU backend with native AVX2 on Kaggle x86)
print('\n--- Building ggml ---')
t0 = time.time()
build_dir = f'{LAMIO_DIR}/build'
os.makedirs(build_dir, exist_ok=True)

# Configure with native CPU optimizations (AVX2/FMA on Kaggle)
result = subprocess.run([
    'cmake', '-B', build_dir, '-S', LAMIO_DIR,
    '-DCMAKE_BUILD_TYPE=Release',
    '-DGGML_NATIVE=ON',
    '-DGGML_AVX2=ON',
    '-DGGML_FMA=ON',
    '-DGGML_F16C=ON',
    '-DGGML_AVX=ON',
    '-DLLAMA_CURL=OFF',
    '-DBUILD_SHARED_LIBS=ON',
], capture_output=True, text=True)
print(result.stdout[-500:] if result.stdout else '')
if result.returncode != 0:
    print('CMAKE ERROR:', result.stderr[-1000:])

# Build just ggml (parallel, limited jobs to avoid OOM)
result = subprocess.run([
    'cmake', '--build', build_dir, '-j', '4',
    '--target', 'ggml',
], capture_output=True, text=True, timeout=300)
print(f'ggml build: {time.time()-t0:.0f}s, exit={result.returncode}')
if result.returncode != 0:
    print('BUILD ERROR:', result.stderr[-1000:])

# Build Lamio v2
print('\n--- Building Lamio v2 ---')
t0 = time.time()
v2_build = f'{LAMIO_DIR}/v2/build'
os.makedirs(v2_build, exist_ok=True)
result = subprocess.run(['cmake', '-B', v2_build, '-S', f'{LAMIO_DIR}/v2'], capture_output=True, text=True)
print(result.stdout[-300:] if result.stdout else '')
result = subprocess.run(['cmake', '--build', v2_build, '-j', '4'], capture_output=True, text=True, timeout=120)
print(f'Lamio v2 build: {time.time()-t0:.0f}s, exit={result.returncode}')
if result.returncode != 0:
    print('BUILD ERROR:', result.stderr[-1000:])

# Verify binary exists
lamio_bin = f'{v2_build}/src/lamio'
if os.path.exists(lamio_bin):
    os.chmod(lamio_bin, 0o755)
    print(f'\nLamio binary: {lamio_bin}')
    subprocess.run([lamio_bin, '--info', '/dev/null'], capture_output=True, timeout=5)
    print('Binary OK')
else:
    print(f'ERROR: {lamio_bin} not found!')


In [ ]:
#@title Download Ornith-35B GGUF (Q4_K_M) to /tmp/models
import subprocess, os, time

MODEL_URL = 'https://huggingface.co/ornith-ai/Ornith-1.0-35B-GGUF/resolve/main/ornith-1.0-35b-Q4_K_M.gguf'
MODEL_PATH = '/tmp/models/Ornith-35B-Q4_K_M.gguf'

if os.path.exists(MODEL_PATH) and os.path.getsize(MODEL_PATH) > 18 * 1024**3:
    print(f'Model already exists: {os.path.getsize(MODEL_PATH) / 1024**3:.1f} GB')
else:
    print(f'Downloading Ornith-35B Q4_K_M to {MODEL_PATH}...')
    t0 = time.time()
    # Use wget with progress bar (follows redirects)
    proc = subprocess.Popen([
        'wget', '-q', '--show-progress', '-O', MODEL_PATH, MODEL_URL
    ], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    # Monitor download progress
    while proc.poll() is None:
        if os.path.exists(MODEL_PATH):
            size_gb = os.path.getsize(MODEL_PATH) / 1024**3
            elapsed = time.time() - t0
            speed = size_gb / max(elapsed, 1)
            print(f'\r  {size_gb:.1f} GB / {elapsed:.0f}s ({speed:.1f} GB/s)', end='', flush=True)
        time.sleep(3)
    proc.wait()
    elapsed = time.time() - t0
    final_size = os.path.getsize(MODEL_PATH) / 1024**3 if os.path.exists(MODEL_PATH) else 0
    print(f'\nDownload complete: {final_size:.1f} GB in {elapsed:.0f}s')

print(f'Model: {MODEL_PATH} ({os.path.getsize(MODEL_PATH) / 1024**3:.1f} GB)')


In [ ]:
#@title Model Info + RAM Baseline
import subprocess, os, psutil

LAMIO_BIN = '/tmp/lamio/v2/build/src/lamio'
MODEL_PATH = '/tmp/models/Ornith-35B-Q4_K_M.gguf'
ENV = {'LD_LIBRARY_PATH': '/tmp/lamio/build/bin'}

print('=== RAM BEFORE ===')
print(f'  Available: {psutil.virtual_memory().available / 1024**3:.1f} GB')
print(f'  Used: {psutil.virtual_memory().used / 1024**3:.1f} GB')

print('\n=== MODEL INFO ===')
result = subprocess.run([LAMIO_BIN, '--info', MODEL_PATH], env=ENV, capture_output=True, text=True, timeout=30)
print(result.stdout)


In [ ]:
#@title Run Inference (with ExpertRouter eviction)
import subprocess, os, time, psutil

LAMIO_BIN = '/tmp/lamio/v2/build/src/lamio'
MODEL_PATH = '/tmp/models/Ornith-35B-Q4_K_M.gguf'
ENV = dict(os.environ, LD_LIBRARY_PATH='/tmp/lamio/build/bin')

PROMPT = '<|im_start|>user\nHi<|im_end|>\n<|im_start|>assistant\n'

# Run with ExpertRouter budget + threads + auto-stop
cmd = [LAMIO_BIN, MODEL_PATH,
       '--generate', '--prompt', PROMPT,
       '--n-gen', '128', '--temp', '0.3',
       '--threads', str(os.cpu_count()),
       '--max-rss-mb', '4096',
       '--auto-stop']

print(f'Running: {" ".join(cmd[:3])} ... --max-rss-mb 4096 --auto-stop')
print(f'CPU threads: {os.cpu_count()}')
print(f'\n=== RAM BEFORE ===')
print(f'  Available: {psutil.virtual_memory().available / 1024**3:.1f} GB')

t0 = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, env=ENV)

# Monitor RSS during inference
peak_rss = 0
while proc.poll() is None:
    try:
        with open(f'/proc/{proc.pid}/status') as f:
            for line in f:
                if line.startswith('VmRSS:'):
                    kb = int(line.split()[1])
                    peak_rss = max(peak_rss, kb)
                    break
    except FileNotFoundError:
        break
    time.sleep(2)

stdout, stderr = proc.communicate()
elapsed = time.time() - t0

# Parse output
pieces = [l for l in stdout.splitlines() if l.startswith('piece:')]
text = ''.join(l.replace('piece:', '', 1) for l in pieces)

print(f'\n=== RESULT ===')
print(f'Tokens: {len(pieces)}')
print(f'Time: {elapsed:.1f}s')
if len(pieces) > 0:
    print(f'Speed: {len(pieces)/elapsed:.2f} t/s')
print(f'Peak RSS: {peak_rss/1024:.0f} MB')
print(f'\nOutput: {text[:300]}')

# Check ExpertRouter logs
evict_lines = [l for l in stderr.splitlines() if 'expert_router' in l.lower()]
if evict_lines:
    print(f'\n=== ExpertRouter ===')
    for l in evict_lines[:10]:
        print(f'  {l}')
else:
    print('\nNo ExpertRouter activity (RSS under budget)')

print(f'\n=== RAM AFTER ===')
print(f'  Available: {psutil.virtual_memory().available / 1024**3:.1f} GB')
print(f'  Used: {psutil.virtual_memory().used / 1024**3:.1f} GB')


In [ ]:
#@title Start Lamio HTTP Server + Keep Alive
import subprocess, os, time, signal, sys, requests

LAMIO_DIR = '/tmp/lamio'
SERVER_PY = f'{LAMIO_DIR}/v2/server.py'
MODEL_NAME = 'Ornith-35B-Q4_K_M.gguf'

env = dict(os.environ,
    LD_LIBRARY_PATH=f'{LAMIO_DIR}/build/bin',
    LAMIO_MAX_RSS_MB='4096',
    LAMIO_THREADS=str(os.cpu_count()),
)

# Kill any existing server
subprocess.run(['pkill', '-f', 'server.py'], capture_output=True)
time.sleep(1)

# Start server in background
proc = subprocess.Popen([
    'python3', SERVER_PY,
], cwd=LAMIO_DIR + '/v2', env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

# Wait for server to start
for i in range(15):
    time.sleep(2)
    try:
        r = requests.get('http://0.0.0.0:5180/api/health', timeout=3)
        if r.status_code == 200:
            print(f'Server online: {r.json()}')
            break
    except:
        pass
else:
    print('Server failed to start')
    sys.exit(1)

# Load the model
print(f'Loading model: {MODEL_NAME}')
r = requests.post('http://0.0.0.0:5180/api/models/load', json={'model': MODEL_NAME}, timeout=10)
print(f'Load response: {r.json()}')

# Chat test
print('\n=== Chat Test ===')
r = requests.post('http://0.0.0.0:5180/v1/chat/completions', json={
    'messages': [{'role': 'user', 'content': 'What is 2+2?'}],
    'max_tokens': 64, 'temperature': 0.3, 'stream': True
}, stream=True, timeout=600)

reply = ''
for line in r.iter_lines():
    line = line.decode('utf-8', errors='ignore').strip()
    if line.startswith('data: ') and '[DONE]' not in line:
        import json
        try:
            chunk = json.loads(line[6:])
            delta = chunk.get('choices', [{}])[0].get('delta', {}).get('content', '')
            if delta: reply += delta
        except: pass

print(f'Reply: {reply[:200]}')

# Keep server alive
print('\nServer running on http://0.0.0.0:5180')
print('Keeping kernel alive...')
while True:
    time.sleep(60)
    try:
        r = requests.get('http://0.0.0.0:5180/api/health', timeout=5)
        print(f'  Health: {r.json()["status"]} | RAM: {__import__("psutil").virtual_memory().available/1024**3:.1f}GB free')
    except:
        print('  Server may have crashed, restarting...')
        break
